In [0]:
# ============================================================
# CHESS LIVE ANALYTICS PROJECT
# SILVER LAYER CONFIGURATION
# ============================================================

BRONZE_TABLE = "grandmasters.bronze.tv_feed"

SILVER_GAMES = "grandmasters.silver.game_events"
SILVER_MOVES = "grandmasters.silver.game_move_events"
SILVER_PLAYER_METRICS = "grandmasters.silver.player_metrics"
SILVER_PLAYER_MOVES = "grandmasters.silver.player_moves"

In [0]:
game_events_df = spark.table(SILVER_GAMES)
player_moves_df = spark.table(SILVER_PLAYER_MOVES)

In [0]:
from pyspark.sql import functions as F

player_metrics_df = (
    player_moves_df
    .join(game_events_df.select("game_id", "white_player", "black_player"), on="game_id", how="left")
    .withColumn("player_name", F.when(F.col("player") == "White", F.col("white_player")).otherwise(F.col("black_player")))
    .groupBy("player_name")
    .agg(
        F.count("*").alias("total_moves"),
        F.countDistinct("game_id").alias("games_played"),
        F.avg("think_time_secs").alias("avg_think_time_secs"),
        F.max("think_time_secs").alias("max_think_time_secs"),
        F.sum("clock_loss_secs").alias("total_clock_loss_secs"),
        F.avg("remaining_clock_secs").alias("avg_remaining_clock_secs")
    )
)

player_metrics_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(SILVER_PLAYER_METRICS)

print(f"Written to {SILVER_PLAYER_METRICS}")

display(player_metrics_df)